In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch 
from torch.utils.data import DataLoader
import sys
sys.path.append('../PyTorch-Pretrained-ViT/')
sys.path.append('../../datasets/')
from sequences import SequencesDataset
from pytorch_pretrained_vit import ViT
import torchvision
from torch import nn
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
model =  ViT('B_16_imagenet1k', pretrained = True)

In [ ]:
order = 4
exposure = 4
seq_length = 1
train_dataset = SequencesDataset(verbose = False, sequence_length = 3, contour_parameter_maps_root_folder ="/media/cmu/DATA1/francis/Collaborations/ARL/ExperimentalDepthPrediction/datasets/contour_parameter_maps", image_type = 'Temp',  thermal_root_folder = '../../datasets/sequence')

In [ ]:
resize = torchvision.transforms.Resize((384,384))#train_dataset.__getitem__(1), (384))

In [ ]:

train_dataloader =  DataLoader(train_dataset, batch_size=2, shuffle=True)

In [ ]:
model(resize(iter(train_dataloader).next()[0].float())).shape

In [ ]:
class VitFineTune(nn.Module):
    def __init__(self,  num_features = 4):
        super(VitFineTune, self).__init__()
        self.vit = ViT('B_16_imagenet1k', pretrained = True)
        self.fc3 = nn.Linear(1000, 100)
        self.fc4 = nn.Linear(100, num_features)
        self.resize= torchvision.transforms.Resize((384,384))
    def forward(self, x):
        embedding = self.vit(self.resize(x))
        x = embedding.view(x.shape[0], -1)        
        # x = F.relu(self.fc1(x))
        # x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))

        x = self.fc4(x)
        return x

In [ ]:
exposure = 4
order = 3
seq_length = 3
train_dataset = SequencesDataset(exposure = exposure, polynomial_order = order,split = 'train', sequence_length = seq_length)
test_dataset = SequencesDataset(exposure = exposure, polynomial_order = order, split= 'test', sequence_length = seq_length)
dev_dataset = SequencesDataset(exposure = exposure, polynomial_order = order, split = 'validation', sequence_length = seq_length)

train_dataloader =  DataLoader(train_dataset, batch_size=2, shuffle=True)
test_dataloader =  DataLoader(test_dataset, batch_size=2, shuffle=True)
dev_dataloader =  DataLoader(dev_dataset, batch_size=2, shuffle=True)

In [ ]:
import datetime
from os import path
import os
now = datetime.datetime.now()
datetime_string = now.strftime("%Y_%m_%d_%H_%M_%S")
if os.getcwd().endswith('notebooks'):
    results_folder = path.join(os.getcwd(), '../../', f'runs/simple_transformer/exposure_{exposure}/order_{order}_frames_{seq_length}/{datetime_string}')
else:
    results_folder = 'runs/simple_transformer/exposure_{exposure}/order_{order}_frames_{seq_length}/{datetime_string}'
os.makedirs(results_folder, exist_ok=True)

In [ ]:
device = 'cuda'
criterion  = nn.MSELoss()
learning_rate = 3e-5
losses =[]
test_losses = []
import time
import os
model = VitFineTune(2*(order + 1))
model.to(device)
plotting_interval = 5
epochs = 1000
from tqdm import tqdm
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate,weight_decay = 0)
for epoch in tqdm(range(epochs)):
    model.train()
    running_loss = 0
    test_running_loss = 0
    running_scaled_loss = 0
    start_time = time.time()
    model.train()
    
    for batch_num, (image, target) in enumerate(train_dataloader):
        oldtime = time.time()
        print(batch_num, len(train_dataloader))
        image = image.to(device)
        target = target.to(device)
        output = model((image.float()))
        loss = criterion(target.float(), (output.float()))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()/len(train_dataloader)
        print(f'time elapsed{time.time() - oldtime}')
    losses.append(running_loss)
    model.eval()
    for batch_num, (image, target) in enumerate(dev_dataloader):
        image = image.to(device)
        target = target.to(device)
        output = model((image.float()))
        test_loss = criterion(target.float(), (output.float()))
        test_running_loss += test_loss.item()/len(test_dataloader)

    test_losses.append(test_running_loss)

    if epoch % 20 == 0:
        print(f'Epoch [{epoch}/{epochs}]Train Loss: {losses[-1]}, Validation Loss: {test_losses[-1]}')
        np.savetxt(os.path.join(results_folder, 'test_loss.txt'), test_losses)
        np.savetxt(os.path.join(results_folder, 'train_loss.txt'), losses)
        states = [
            model.state_dict(),
            optimizer.state_dict(),
            epoch,
            batch_num,
        ]
        torch.save(states, os.path.join(results_folder, "ckpt.pth"))
